In [ ]:
import sqlite3
from pathlib import Path
import pandas as pd

RUTA_CSV = Path("data/conjunto_de_datos_inpc_mensual.csv")
RUTA_DB  = Path("data/inpc.db")
RUTA_SQL = Path("01_crear_tablas.sql")

# 1) Leemos archivo con el formato INEGI (csv o exceel)
df = pd.read_csv(RUTA_CSV, encoding="utf-8-sig", low_memory=False)
df

# 2) Adaptamos formato de las fechas
df["fecha"] = pd.to_datetime(df["FECHA"], dayfirst=True).dt.strftime("%Y-%m-%d")
df["VALOR"] = pd.to_numeric(df["VALOR"], errors="coerce")
assert df["VALOR"].notna().all(), "Hay valores no numéricos en VALOR"

df["etiqueta"] = df["CONCEPTO"].str.split(", ").str[-1]
catalogo = pd.DataFrame([
    (1, "Precios al Consumidor (INPC)", "INPC general",            "General",        1),
    (2, "Subyacente",                   "Subyacente",              "Subyacente",     1),
    (3, "No subyacente",                "No subyacente",           "No subyacente",  1),
    (4, "Mercancías",                   "Mercancías",              "Subyacente",     0),
    (5, "Servicios",                    "Servicios",               "Subyacente",     0),
    (6, "Agropecuarios",                "Agropecuarios",           "No subyacente",  0),
    (7, "Energéticos y tarifas autorizadas por el gobierno",
                                        "Energéticos y tarifas",   "No subyacente",  0),
], columns=["concepto_id", "etiqueta", "nombre", "grupo", "es_agregado"])

df = df.merge(catalogo[["etiqueta", "concepto_id"]],
              on="etiqueta", how="left", validate="many_to_one")
assert df["concepto_id"].notna().all(), "Alguna etiqueta no está en el catálogo"

conn = sqlite3.connect(RUTA_DB)
conn.execute("PRAGMA foreign_keys = ON;")
conn.executescript(RUTA_SQL.read_text(encoding="utf-8"))

conceptos = list(catalogo[["concepto_id", "nombre", "grupo", "es_agregado"]]
                 .itertuples(index=False, name=None))
conn.executemany(
    "INSERT INTO conceptos (concepto_id, nombre, grupo, es_agregado) VALUES (?, ?, ?, ?);",
    conceptos,
)
precios = list(zip(df["fecha"], df["concepto_id"].astype(int), df["VALOR"].astype(float)))
conn.executemany(
    "INSERT INTO precios (fecha, concepto_id, indice) VALUES (?, ?, ?);",
    precios,
)
conn.commit()
print(f"Insertados: {len(precios)} precios y {len(conceptos)} conceptos.")

# 7) Validación
checks = {
    "total precios":            "SELECT COUNT(*) FROM precios;",
    "conceptos distintos":      "SELECT COUNT(DISTINCT concepto_id) FROM precios;",
    "rango de fechas":          "SELECT MIN(fecha), MAX(fecha) FROM precios;",
    "duplicados (debe ser 0)":  """SELECT COUNT(*) FROM (
                                     SELECT fecha, concepto_id FROM precios
                                     GROUP BY fecha, concepto_id HAVING COUNT(*) > 1);""",
    "registros por concepto":   """SELECT c.nombre, COUNT(*) FROM precios p
                                     JOIN conceptos c USING (concepto_id)
                                     GROUP BY c.nombre ORDER BY c.nombre;""",
}
for etiqueta, query in checks.items():
    print(f"\n{etiqueta}:")
    for fila in conn.execute(query):
        print("  ", fila)

conn.close()

Insertados: 2002 precios y 7 conceptos.

total precios:
   (2002,)

conceptos distintos:
   (7,)

rango de fechas:
   ('2003-01-01', '2026-10-01')

duplicados (debe ser 0):
   (0,)

registros por concepto:
   ('Agropecuarios', 286)
   ('Energéticos y tarifas', 286)
   ('INPC general', 286)
   ('Mercancías', 286)
   ('No subyacente', 286)
   ('Servicios', 286)
   ('Subyacente', 286)
